<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day06-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 6 lab: linear regression vs. a small MLP, on a real biological target {.unnumbered}

A teaser, not a full machine-learning treatment -- Day 8 covers the
fundamentals (train/validation/test splits, evaluation) properly, and
Day 9 covers regularization and architecture choices. Here, just:
assemble one small real dataset, fit two models on it the way this
page's own "Learning the weights" and "Adding one hidden layer fixes
it" sections already did on toy data, and see which one actually
predicts better.

**The task**: predict a real protein's real alpha-helix fraction from
its real amino-acid composition. This page's own HBB example already
established the anchor fact -- 117/147 residues (~80%) of HBB are
alpha-helix, fetched live from EBI's PDBe API. Here, the same live API
is used across 55 more real, structurally diverse proteins (not more
near-duplicate globins -- a real range of folds, from almost no helix
to mostly helix) to build a real dataset: 20 features per protein (the
fraction of each amino acid in its sequence), one real target per
protein (its real helix fraction).


## 1. Assemble the real dataset

55 real PDB entries (picked by an automated, resolution/length-filtered
search of the RCSB, spread across many recent depositions rather than
near-duplicate mutants of the same protein -- fixed here as an explicit
list so this cell runs the same way every time, but every value below
is still fetched live, not hand-typed). For each: fetch its real
sequence and real secondary-structure assignment from EBI's PDBe API
(the exact same endpoints the book page's own HBB example uses) and
compute its real amino-acid composition and real helix fraction.


In [ ]:
import requests
import numpy as np

PDB_IDS = ['21XG', '38LJ', '9WRU', '9NVT', '9XZ9', '7ILS', '9RYL', '9TLR', '32TC', '13DZ', '9TCX', '9YBX', '9N2Y', '29OL', '9PNX', '9YMO', '9HWE', '9RPQ', '9RR6', '9W2A', '21LO', '9RK7', '9OU3', '9P39', '9T4I', '9VOM', '9W28', '9RFG', '9V5C', '9RQP', '9I4O', '9QWL', '9QYO', '9WM8', '9QRF', '9X6R', '9TZG', '9QGV', '9U97', '9YK5', '9IC9', '9XPD', '9DJF', '9DJR', '9NK4', '9NCE', '9I1B', '9RQQ', '7IPN', '9LCS', '9Z70', '9KZN', '9SI4', '9P45', '9FD0']
AAS = list("ACDEFGHIKLMNPQRSTVWY")

def fetch_protein(pdbid):
    lo = pdbid.lower()
    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/molecules/{lo}", timeout=15)
    r.raise_for_status()
    polymers = [m for m in r.json()[lo] if m.get("molecule_type") == "polypeptide(L)" and m.get("sequence")]
    mol = polymers[0]
    seq, chain_id, length = mol["sequence"], mol["in_chains"][0], mol["length"]

    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/secondary_structure/{lo}", timeout=15)
    r.raise_for_status()
    helices = None
    for molecule in r.json()[lo]["molecules"]:
        for chain in molecule["chains"]:
            if chain["chain_id"] == chain_id:
                helices = chain["secondary_structure"].get("helices", [])
    helix_residues = sum(h["end"]["residue_number"] - h["start"]["residue_number"] + 1 for h in helices)
    helix_frac = helix_residues / length

    composition = [seq.count(aa) / len(seq) for aa in AAS]
    return composition, helix_frac, length

X_rows, y_rows, lengths = [], [], []
for pdbid in PDB_IDS:
    comp, frac, length = fetch_protein(pdbid)
    X_rows.append(comp)
    y_rows.append(frac)
    lengths.append(length)

X = np.array(X_rows)
y = np.array(y_rows)

print(f"Real dataset: {X.shape[0]} real proteins, {X.shape[1]} real features (amino-acid composition)")
print(f"Real helix fraction: min={y.min():.2f}, max={y.max():.2f}, mean={y.mean():.2f}")
print(f"Real sequence lengths: min={min(lengths)}, max={max(lengths)}")


## 2. Holding out real data to check on

A fair comparison needs to test each model on real proteins it never
saw while fitting -- otherwise a model could just be memorizing, not
predicting. Day 8 covers this properly (train/validation/test, why a
single split can be a noisy estimate, cross-validation); here, one
simple holdout is enough for a teaser: fit both models on 75% of the
real data, check both on the same held-out 25% they never saw.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(f"Training on {len(X_train)} real proteins, testing on {len(X_test)} real proteins never used for fitting.")


## 3. Linear regression

Exactly the same tool this page's own "Learning the weights, instead
of deriving them" section already introduced (closed-form, via the
normal equations) -- just with 20 real features instead of 1 toy one.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

linreg = LinearRegression().fit(X_train, y_train)
pred_linreg = linreg.predict(X_test)
mse_linreg = mean_squared_error(y_test, pred_linreg)

print(f"Linear regression: held-out MSE = {mse_linreg:.4f}")


## 4. A small multilayer perceptron, from scratch

The same raw-tensors, plain-gradient-descent style as the XOR notebook's
own hidden-layer model -- one small fixed hidden layer (8 units, `tanh`),
trained by hand with no `nn.Module`, no optimizer class, no
regularization (Day 9's territory, not today's). Only the loss changes:
mean squared error for a real number, not binary cross-entropy for a
yes/no class.


In [ ]:
import torch

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
Xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(0)
hidden = 8
W1 = (torch.randn(20, hidden) * 0.3).requires_grad_()
b1 = torch.zeros(hidden, requires_grad=True)
W2 = (torch.randn(hidden, 1) * 0.3).requires_grad_()
b2 = torch.zeros(1, requires_grad=True)
params = [W1, b1, W2, b2]

lr_rate = 0.05
epoch_losses = []
for epoch in range(3000):
    hid = torch.tanh(Xtr @ W1 + b1)
    pred = hid @ W2 + b2
    loss = torch.nn.functional.mse_loss(pred, ytr)
    loss.backward()
    with torch.no_grad():
        for p in params:
            p -= lr_rate * p.grad
            p.grad.zero_()
    epoch_losses.append(loss.item())

with torch.no_grad():
    hid_test = torch.tanh(Xte @ W1 + b1)
    pred_mlp = hid_test @ W2 + b2
    mse_mlp = torch.nn.functional.mse_loss(pred_mlp, yte).item()

print(f"MLP (8-unit hidden layer): held-out MSE = {mse_mlp:.4f}")
print(f"Training MSE: {epoch_losses[0]:.4f} (epoch 0) -> {epoch_losses[-1]:.4f} (epoch 3000)")


## 5. The training curve

Same "adapt the loop to record MSE every epoch and plot it" exercise
that already worked on the from-scratch models above -- here shown
directly rather than left as an exercise, since this is a teaser.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(epoch_losses)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.title("MLP training loss, real held-out MSE = {:.4f}".format(mse_mlp))
plt.tight_layout()
plt.show()


## 6. So which one actually predicted better?

Report both real numbers honestly -- there's no guarantee a hidden
layer wins here, and small, largely-additive biological features (amino
acid composition, in this case) are exactly the regime where a plain
linear model can hold its own. This is a real result on a real, small
dataset, not a rigged demonstration.


In [ ]:
print(f"Linear regression held-out MSE: {mse_linreg:.4f}")
print(f"MLP (8-unit hidden layer) held-out MSE: {mse_mlp:.4f}")
print()
winner = "Linear regression" if mse_linreg < mse_mlp else "The MLP"
print(f"{winner} had the lower real held-out error on this real dataset.")


## Optional: does every feature help?

The old version of this exercise (from a previous year of this course)
asked exactly this question about a different toy dataset: try a
smaller subset of the 20 real features instead of all of them, and see
whether training/prediction changes. Not required -- just worth poking
at if you're curious.


In [ ]:
# Try just a handful of amino acids instead of all 20, e.g. the ones
# classically associated with helix formation/breaking (Day 6's own
# Chou-Fasman mention): alanine, glutamate, leucine, methionine (helix
# formers) and proline, glycine (helix breakers).
subset = ["A", "E", "L", "M", "P", "G"]
idx = [AAS.index(aa) for aa in subset]

X_train_sub = X_train[:, idx]
X_test_sub = X_test[:, idx]

linreg_sub = LinearRegression().fit(X_train_sub, y_train)
mse_sub = mean_squared_error(y_test, linreg_sub.predict(X_test_sub))

print(f"Linear regression, all 20 features:      held-out MSE = {mse_linreg:.4f}")
print(f"Linear regression, these 6 features only: held-out MSE = {mse_sub:.4f}")


## Done

Once every cell above has run and printed real output, answer the Day
6 lab quiz on Canvas using your own real numbers.
